**AE + GA**

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx') 
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Autoencoder-Classifier Model

class JointAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointAutoencoderClassifier, self).__init__()
        #Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, latent_dim),
        )
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh() 
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstruction = self.decoder(latent)
        logits = self.classifier(latent)
        return reconstruction, logits

# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance losses
lambda_recon = 0.5

def combined_loss(reconstructed, original, logits, labels):
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    loss_class = classification_loss_fn(logits, labels)
    return lambda_recon * loss_recon + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32


# Initialize model, optimizer, and scheduler
model = JointAutoencoderClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50

# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits = model(inputs)
        loss = combined_loss(reconstruction, inputs, logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")

# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_latent = model.encoder(X_train).numpy()
    test_latent = model.encoder(X_test).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

from deap import base, creator, tools, algorithms
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import random
import warnings
warnings.filterwarnings("ignore")

print("Starting GA")
print("latent data:", train_latent.shape)

# Define all the needed parameters

LATENT_FEATURES = train_latent.shape[1]
N_GENERATIONS = 50
POPULATION_SIZE = 100
P_CROSSOVER = 0.7
P_MUTATION = 0.2
TOURNAMENT_SIZE = 3
N_SELECTED_LATENT_FEATURES = 10

try:
    del creator.FitnessMax
    del creator.Individual
except:
    pass

# Create types for GA
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=LATENT_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual_with_latent(individual):
    # Get selected latent features
    selected_indices = [i for i, val in enumerate(individual) if val == 1]
    
    # If too few features are selected, penalize the individual
    if len(selected_indices) < 2:
        return 0.0,
    
    # Create a modified input tensor with zeros for non-selected features
    modified_latent = train_latent[:, selected_indices]
    
    # Train a simple classifier on the selected latent features
    classifier = GaussianNB()
    scores = cross_val_score(classifier, modified_latent, y_train_np, cv=5)
    accuracy = scores.mean()

    return accuracy,

toolbox.register("evaluate", evaluate_individual_with_latent)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

def run_ga():
    # Initialize population
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Statistics to track progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Hall of fame to keep track of best individuals
    hof = tools.HallOfFame(1)
    
    # Running the algorithm
    population, logbook = algorithms.eaSimple(
        population, 
        toolbox, 
        cxpb=P_CROSSOVER, 
        mutpb=P_MUTATION, 
        ngen=N_GENERATIONS, 
        stats=stats, 
        halloffame=hof, 
        verbose=True
    )
    
    return population, logbook, hof

'''
import time
print("Starting Genetic Algorithm for latent feature selection")
start_time = time.time()
population, logbook, hof = run_ga()
end_time = time.time()
print(f"GA completed in {(end_time - start_time)/60:.2f} minutes")'''

population, logbook, hof = run_ga()

# Get the best individual
best_individual = hof[0]
selected_latent_features = [i for i, val in enumerate(best_individual) if val == 1]
num_selected = len(selected_latent_features)

print(f"Number of selected latent features: {num_selected} out of {LATENT_FEATURES}")
print('The selected features: ', selected_latent_features)
print(f"Best fitness: {best_individual.fitness.values[0]}")


#testing the ga selected features on the models
print('Testing GA Selected Features: ')
modified_latent_train = train_latent[:, selected_latent_features]
modified_latent_test = test_latent[:, selected_latent_features]

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(modified_latent_train, y_train_np)
svm_preds = svm_model.predict(modified_latent_test)
svm_acc = accuracy_score(y_test_np, svm_preds)
print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(modified_latent_train, y_train_np)
nbc_preds = nbc_model.predict(modified_latent_test)
nbc_acc = accuracy_score(y_test_np, nbc_preds)
print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(modified_latent_train, y_train_np)
rf_preds = rf_model.predict(modified_latent_test)
rf_acc = accuracy_score(y_test_np, rf_preds)
print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.1988
Epoch [2/50], Loss: 0.7913
Epoch [3/50], Loss: 0.6534
Epoch [4/50], Loss: 0.5332
Epoch [5/50], Loss: 0.5158
Epoch [6/50], Loss: 0.4874
Epoch [7/50], Loss: 0.4726
Epoch [8/50], Loss: 0.4734
Epoch [9/50], Loss: 0.4389
Epoch [10/50], Loss: 0.4809
Epoch [11/50], Loss: 0.4041
Epoch [12/50], Loss: 0.4602
Epoch [13/50], Loss: 0.4252
Epoch [14/50], Loss: 0.4045
Epoch [15/50], Loss: 0.4323
Epoch [16/50], Loss: 0.3552
Epoch [17/50], Loss: 0.4465
Epoch [18/50], Loss: 0.3687
Epoch [19/50], Loss: 0.3868
Epoch [20/50], Loss: 0.4289
Epoch [21/50], Loss: 0.3747
Epoch [22/50], Loss: 0.3949
Epoch [23/50], Loss: 0.3758
Epoch [24/50], Loss: 0.3798
Epoch [25/50], Loss: 0.3838
Epoch [26/50], Loss: 0.3613
Epoch [27/50], Loss: 0.4473
Epoch [28/50], Loss: 0.3815
Epoch [29/50], Loss: 0.3757
Epoch [30/50], Loss: 0.4179
Epoch [31/50], Loss: 0.3815
Epoch [32/50], Loss: 0.3708
Epoch [33/50], Loss: 0.3649
Epoch [34/50], Loss: 0.3546
Epoch [35/50], Loss: 0.4337


**VAE + GA**

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


# 1. Data Loading & Preparation

df = pd.read_excel('minmax.xlsx')
data = df.values
labels = pd.read_csv('idC_with_header.csv').values.flatten() - 1  # Adjust labels to be in range [0, 13]

X_tensor = torch.tensor(data, dtype=torch.float32)
y_tensor = torch.tensor(labels, dtype=torch.long)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 2. Define the Joint Variational Autoencoder-Classifier Model

class JointVAEClassifier(nn.Module):
    def __init__(self, input_dim, latent_dim=32, num_classes=14):
        super(JointVAEClassifier, self).__init__()
        # Encoder
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128)
        )
        # Layers to produce the mean and log-variance for the latent distribution
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Linear(512, input_dim),
            nn.Tanh()  
        )
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def forward(self, x):
        # Encode input to get intermediate representation
        x_encoded = self.encoder_net(x)
        # Produce latent distribution parameters
        mu = self.fc_mu(x_encoded)
        logvar = self.fc_logvar(x_encoded)
        # Sample latent vector
        z = self.reparameterize(mu, logvar)
        # Reconstruct input and classify based on latent vector
        reconstruction = self.decoder(z)
        logits = self.classifier(z)
        return reconstruction, logits, mu, logvar


# 3. Loss Functions & Optimizer

reconstruction_loss_fn = nn.MSELoss()
classification_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

# Hyperparameter to balance reconstruction and KL loss and classification loss
lambda_recon = 0.5

def vae_combined_loss(reconstructed, original, logits, labels, mu, logvar):
    # Reconstruction loss
    loss_recon = reconstruction_loss_fn(reconstructed, original)
    # KL divergence loss: average over batch
    loss_kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    # Classification loss
    loss_class = classification_loss_fn(logits, labels)
    # Combined loss: balance reconstruction (with KL) and classification
    return lambda_recon * (loss_recon + loss_kl) + (1 - lambda_recon) * loss_class

input_dim = data.shape[1]
num_classes = 14
latent_dim = 32

# Initialize model, optimizer, and scheduler
model = JointVAEClassifier(input_dim, latent_dim, num_classes)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
num_epochs = 50


# 4. Joint Training (Pretraining Stage)

print("=== Joint Training Stage ===")
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        reconstruction, logits, mu, logvar = model(inputs)
        loss = vae_combined_loss(reconstruction, inputs, logits, labels, mu, logvar)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


# 5. Extract Latent Features Using the Trained Encoder

model.eval()
with torch.no_grad():
    train_encoded = model.encoder_net(X_train)
    train_latent = model.fc_mu(train_encoded).numpy()
    
    test_encoded = model.encoder_net(X_test)
    test_latent = model.fc_mu(test_encoded).numpy()

# Convert labels to NumPy arrays
y_train_np = y_train.numpy()
y_test_np = y_test.numpy()

# --- Genetic Algorithm Implementation ---

import random
from deap import base, creator, tools, algorithms

print("Starting GA")
print("latent data:", train_latent.shape)

# Define all the needed parameters

LATENT_FEATURES = train_latent.shape[1]
N_GENERATIONS = 50
POPULATION_SIZE = 100
P_CROSSOVER = 0.7
P_MUTATION = 0.2
TOURNAMENT_SIZE = 3
N_SELECTED_LATENT_FEATURES = 10

try:
    del creator.FitnessMax
    del creator.Individual
except:
    pass

# Create types for GA
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=LATENT_FEATURES)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evaluate_individual_with_latent(individual):
    # Get selected latent features
    selected_indices = [i for i, val in enumerate(individual) if val == 1]
    
    # If too few features are selected, penalize the individual
    if len(selected_indices) < 2:
        return 0.0,
    
    # Create a modified input tensor with zeros for non-selected features
    modified_latent = train_latent[:, selected_indices]
    
    # Train a simple classifier on the selected latent features
    classifier = GaussianNB()
    scores = cross_val_score(classifier, modified_latent, y_train_np, cv=5)
    accuracy = scores.mean()

    return accuracy,

toolbox.register("evaluate", evaluate_individual_with_latent)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=TOURNAMENT_SIZE)

def run_ga():
    # Initialize population
    population = toolbox.population(n=POPULATION_SIZE)
    
    # Statistics to track progress
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Hall of fame to keep track of best individuals
    hof = tools.HallOfFame(1)
    
    # Running the algorithm
    population, logbook = algorithms.eaSimple(
        population, 
        toolbox, 
        cxpb=P_CROSSOVER, 
        mutpb=P_MUTATION, 
        ngen=N_GENERATIONS, 
        stats=stats, 
        halloffame=hof, 
        verbose=True
    )
    
    return population, logbook, hof

'''import time
print("Starting Genetic Algorithm for latent feature selection")
start_time = time.time()
population, logbook, hof = run_ga()
end_time = time.time()
print(f"GA completed in {(end_time - start_time)/60:.2f} minutes")'''

population, logbook, hof = run_ga()

# Get the best individual
best_individual = hof[0]
selected_latent_features = [i for i, val in enumerate(best_individual) if val == 1]
num_selected = len(selected_latent_features)

print(f"Number of selected latent features: {num_selected} out of {LATENT_FEATURES}")
print('The selected features: ', selected_latent_features)
print(f"Best fitness: {best_individual.fitness.values[0]}")


#testing the ga selected features on the models
print('Testing GA Selected Features: ')
modified_latent_train = train_latent[:, selected_latent_features]
modified_latent_test = test_latent[:, selected_latent_features]

# --- SVM Classifier ---
svm_model = SVC(random_state=42)
svm_model.fit(modified_latent_train, y_train_np)
svm_preds = svm_model.predict(modified_latent_test)
svm_acc = accuracy_score(y_test_np, svm_preds)
print(f"SVM Accuracy on latent features selected by GA: {svm_acc * 100:.4f}")

# --- Naïve Bayes Classifier ---
nbc_model = GaussianNB()
nbc_model.fit(modified_latent_train, y_train_np)
nbc_preds = nbc_model.predict(modified_latent_test)
nbc_acc = accuracy_score(y_test_np, nbc_preds)
print(f"Naïve Bayes Accuracy on latent features selected by GA: {nbc_acc * 100:.4f}")

# --- Random Forest Classifier ---
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(modified_latent_train, y_train_np)
rf_preds = rf_model.predict(modified_latent_test)
rf_acc = accuracy_score(y_test_np, rf_preds)
print(f"Random Forest Accuracy on latent features selected by GA: {rf_acc * 100:.4f}")

=== Joint Training Stage ===
Epoch [1/50], Loss: 1.5093
Epoch [2/50], Loss: 1.3158
Epoch [3/50], Loss: 1.1711
Epoch [4/50], Loss: 1.0865
Epoch [5/50], Loss: 1.0008
Epoch [6/50], Loss: 0.9809
Epoch [7/50], Loss: 0.9334
Epoch [8/50], Loss: 0.8758
Epoch [9/50], Loss: 0.8538
Epoch [10/50], Loss: 0.8030
Epoch [11/50], Loss: 0.7514
Epoch [12/50], Loss: 0.7891
Epoch [13/50], Loss: 0.7681
Epoch [14/50], Loss: 0.8035
Epoch [15/50], Loss: 0.7486
Epoch [16/50], Loss: 0.7632
Epoch [17/50], Loss: 0.7080
Epoch [18/50], Loss: 0.7054
Epoch [19/50], Loss: 0.7220
Epoch [20/50], Loss: 0.6865
Epoch [21/50], Loss: 0.7160
Epoch [22/50], Loss: 0.6823
Epoch [23/50], Loss: 0.6734
Epoch [24/50], Loss: 0.6937
Epoch [25/50], Loss: 0.6933
Epoch [26/50], Loss: 0.7211
Epoch [27/50], Loss: 0.6726
Epoch [28/50], Loss: 0.6521
Epoch [29/50], Loss: 0.7056
Epoch [30/50], Loss: 0.6859
Epoch [31/50], Loss: 0.6887
Epoch [32/50], Loss: 0.6529
Epoch [33/50], Loss: 0.6935
Epoch [34/50], Loss: 0.6403
Epoch [35/50], Loss: 0.6762
